# SafeVision · Notebook 3 · Accident-text classifier (N2) + similar-accident search (N3)

**Goal:** teach the NLP part what kind of accident a risky situation can lead to, using **real US accident reports** (OSHA Severe Injury Reports, public domain), and find **similar past accidents** for any situation SafeVision sees.

**Key design idea.** An OSHA accident story describes what already happened ("*fell 12 feet from a ladder and fractured his wrist*"). In our app, the text comes from a photo, **before** any accident ("*worker without a hard hat on a ladder next to a crane*"). So we train and test on three versions of the text:

| Text version | What is removed | Why |
|---|---|---|
| **full** | nothing | comparable with published OSHA classifiers [6], [14] |
| **outcome-masked** | injury words (*fractured, amputated, hospitalized, died...*) | the outcome is what we want to predict |
| **context-only** | injury words **and** how-it-happened words (*fell, struck, caught, crushed...*) | closest to a description made from a photo; **this model is used in the app** |

| ID | Question | Model | Main scores |
|---|---|---|---|
| **E5a** | Simple baseline on each text version | TF-IDF + logistic regression | macro-F1, accuracy |
| **E5b** | Transformer on full text (compare with [6]: macro-F1 0.926) | DistilBERT, 3 outputs | macro-F1 per output |
| **E5c** | Transformer on context-only text (app model) | DistilBERT, 3 outputs | macro-F1 per output |
| **E5d** *(optional)* | Other transformers [14] | DeBERTa-v3-small / RoBERTa | macro-F1 |
| **E7** | Do we retrieve similar past accidents? | keyword (BM25) vs meaning (embeddings) search | precision@3 |

**The three outputs (multi-task):** accident type (8 OIICS event groups, as in [6] and [7]), injured body-part group, and severity (amputation or eye loss vs hospitalisation).

**Fair test:** we split **by year**: train on older reports, test on the newest. Paper [6] names a time-based split as future work, and paper [15] shows that results drift over time.

**Runtime:** GPU, about 30-60 min.

## 0. Install libraries
`transformers` (DistilBERT), `scikit-learn` (baseline and scores), `sentence-transformers` + `faiss` (meaning search), `rank-bm25` (keyword search).

In [ ]:
# [colab-only]
!nvidia-smi
%pip install -q transformers scikit-learn sentence-transformers faiss-cpu rank-bm25 openpyxl huggingface_hub

## 1. Configuration (edit this cell only)

Year split, model names, training settings and the word lists used for masking. Settings follow paper [6] where possible (DistilBERT was its best single model; random oversampling of rare classes helped, SMOTE did not).

In [ ]:
# CONFIG
from pathlib import Path
import re, json, random

OUT = Path("/content/safevision_outputs_nb3")
SIR_PAGE = "https://www.osha.gov/severe-injury-reports"   # the download link to the ZIP is read from this page
SIR_LOCAL = Path("/content/sir_data")                    # where the CSV ends up

TRAIN_MAX_YEAR = 2021   # train: 2015-2021
VAL_YEAR = 2022         # validation: 2022 (choose the best epoch)
# test: 2023 and later (never used for training or tuning)

MODEL_MAIN = "distilbert-base-uncased"                   # best single model in [6]
EXTRA_MODELS = []        # e.g. ["microsoft/deberta-v3-small", "roberta-base"] for E5d (optional, [14])
EMB_MODEL = "BAAI/bge-small-en-v1.5"                     # sentence embeddings for similar-accident search

MAX_LEN = 256; EPOCHS = 3; LR = 3e-5; BATCH = 32; SEED = 42
OVERSAMPLE_MIN = 300     # rare accident types are randomly duplicated up to this many training rows [6]

# Words removed for the 'outcome-masked' version (injury / outcome words)
OUTCOME_WORDS = r"amputat\w*|fractur\w*|broke|broken|hospitali[sz]\w*|admitted|died|death|deceased|fatal\w*|killed|" \
                r"laceration\w*|lacerated|injur\w*|burn(?:s|ed|t)?|bruis\w*|concussion\w*|sprain\w*|dislocat\w*|" \
                r"surgery|treated|treatment|stitches|sutures|crushed|severed|lost|loss|eye"
# Extra words removed for the 'context-only' version (how it happened)
MECHANISM_WORDS = r"fell|fall\w*|slip\w*|trip\w*|struck|strike\w*|hit|hitting|caught|pinned|trapped|crush\w*|" \
                  r"electrocut\w*|shock\w*|exploded|explosion|collapsed?|overturn\w*|rolled|ran over|run over|" \
                  r"contact\w*|exposed|exposure|lifting|pulled|pushed"

RES, FIGS, MODELS = OUT / "results", OUT / "figures", OUT / "models"
for p in [OUT, RES, FIGS, MODELS, SIR_LOCAL]:
    p.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

## 2. Results logger
Same format as Notebooks 1 and 2.

In [ ]:
import csv, datetime
import pandas as pd
LOG_CSV = RES / "experiments_log.csv"
LOG_FIELDS = ["experiment", "run", "model", "text_version", "dataset", "split", "target", "metric", "value", "notes", "timestamp"]

def log_result(experiment, run, model, text_version, dataset, split, target, metric, value, notes=""):
    # Append one measured number to the results log.
    new_file = not LOG_CSV.exists()
    with open(LOG_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new_file:
            w.writeheader()
        w.writerow(dict(experiment=experiment, run=run, model=model, text_version=text_version, dataset=dataset,
                        split=split, target=target, metric=metric, value=round(float(value), 4), notes=notes,
                        timestamp=datetime.datetime.now().isoformat(timespec="seconds")))

def read_log():
    return pd.read_csv(LOG_CSV) if LOG_CSV.exists() else pd.DataFrame(columns=LOG_FIELDS)

## 3. Download the OSHA Severe Injury Reports

OSHA publishes every severe injury report (amputation, hospitalisation, loss of an eye) since 1 January 2015 as one CSV inside a ZIP. The cell reads the ZIP link from the OSHA page. If that fails, download the ZIP yourself from https://www.osha.gov/severe-injury-reports ("Download the full SIR data set") and upload it when asked.

In [ ]:
# [colab-only]
import requests, zipfile, io
try:
    html = requests.get(SIR_PAGE, timeout=60, headers={"User-Agent": "Mozilla/5.0"}).text
    link = re.search(r'href="([^"]+\.zip)"', html).group(1)
    link = link if link.startswith("http") else "https://www.osha.gov" + link
    print("Downloading", link)
    data = requests.get(link, timeout=300, headers={"User-Agent": "Mozilla/5.0"}).content
except Exception as e:
    from google.colab import files
    print("Automatic download failed (", e, "). Please upload the SIR ZIP or CSV:")
    up = files.upload(); data = list(up.values())[0]
if data[:2] == b"PK":
    zipfile.ZipFile(io.BytesIO(data)).extractall(SIR_LOCAL)
else:
    (SIR_LOCAL / "sir.csv").write_bytes(data)
print(list(SIR_LOCAL.rglob("*")))

## 4. Load the data and keep construction only

Column names change a little between OSHA releases, so columns are found by **name similarity** (e.g. "Final Narrative", "EventTitle", "Primary NAICS"). We keep rows whose NAICS industry code starts with **23 (construction)** and that have a narrative.

In [ ]:
csv_path = sorted(SIR_LOCAL.rglob("*.csv"), key=lambda p: p.stat().st_size)[-1]
raw = pd.read_csv(csv_path, encoding="latin-1", low_memory=False)
print(csv_path.name, raw.shape)

def find_col(df, *keys):
    # Return the first column whose simplified name contains all the given key words.
    simple = {c: re.sub(r"[^a-z]", "", c.lower()) for c in df.columns}
    for c, s in simple.items():
        if all(k in s for k in keys):
            return c
    raise KeyError(f"No column with {keys}. Columns: {list(df.columns)}")

COL = dict(date=find_col(raw, "eventdate"), naics=find_col(raw, "naics"), narrative=find_col(raw, "narrative"),
           event=find_col(raw, "event") if "Event" not in raw.columns else "Event",
           event_title=find_col(raw, "eventtitle"),
           body=find_col(raw, "partofbody") if "Part of Body" not in raw.columns else "Part of Body",
           body_title=find_col(raw, "partofbodytitle"), hosp=find_col(raw, "hospitalized"),
           amp=find_col(raw, "amputation"))
if any("eye" in c.lower() for c in raw.columns):        # 'Loss of Eye' exists in newer releases only
    COL["eye"] = find_col(raw, "eye")
else:
    raw["_no_eye"] = 0; COL["eye"] = "_no_eye"
print(json.dumps(COL, indent=1))

df = raw[[*COL.values()]].copy()
df.columns = list(COL.keys())
df = df[df.naics.astype(str).str.startswith("23")]
df = df.dropna(subset=["narrative"])
df["narrative"] = df.narrative.astype(str).str.strip()
df = df[df.narrative.str.len() > 20].drop_duplicates(subset=["narrative"])
df["year"] = pd.to_datetime(df.date, errors="coerce").dt.year
df = df.dropna(subset=["year"]); df["year"] = df.year.astype(int)
print("Construction reports with a narrative:", len(df), "| years", df.year.min(), "-", df.year.max())

## 5. Build the three labels

- **Accident type**: the first digit of the OIICS *event* code gives the 8 groups used by [6] and [7].
- **Body-part group**: the first digit of the OIICS *part of body* code (head, neck, trunk, arms, legs, body systems, multiple parts, other). The printout lists example titles per group so you can confirm the mapping.
- **Severity**: `severe` (amputation or loss of an eye) vs `hospitalised`.

In [ ]:
EVENT_GROUPS = {"1": "Violence by persons or animals", "2": "Transportation incident", "3": "Fire or explosion",
                "4": "Fall, slip or trip", "5": "Exposure to harmful substance or environment",
                "6": "Contact with objects or equipment", "7": "Overexertion or bodily reaction", "9": "Non-classifiable"}
BODY_GROUPS = {"1": "Head", "2": "Neck", "3": "Trunk", "4": "Upper limbs", "5": "Lower limbs",
               "6": "Body systems", "8": "Multiple body parts", "9": "Other or unspecified"}

def first_digit(x):
    # First digit of an OIICS code stored as number or text ('' if missing).
    s = re.sub(r"\D", "", str(x).split(".")[0])
    return s[0] if s else ""

df["event_group"] = df.event.map(first_digit).map(EVENT_GROUPS)
df["body_group"] = df.body.map(first_digit).map(BODY_GROUPS)
df["severity"] = ((pd.to_numeric(df.amp, errors="coerce").fillna(0) > 0) |
                  (pd.to_numeric(df.eye, errors="coerce").fillna(0) > 0)).map({True: "severe (amputation/eye loss)", False: "hospitalised"})
df = df.dropna(subset=["event_group", "body_group"])

for g in sorted(df.body_group.unique()):
    print(f"{g:22s} e.g. {list(df[df.body_group == g].body_title.value_counts().index[:3])}")

TARGETS = {"event_group": sorted(df.event_group.unique()),
           "body_group": sorted(df.body_group.unique()),
           "severity": sorted(df.severity.unique())}
label_counts = pd.concat({t: df[t].value_counts() for t in TARGETS}, axis=0).rename("reports")
label_counts.to_csv(RES / "osha_label_counts.csv")
print(label_counts)

## 6. Make the three text versions (full, outcome-masked, context-only)

Removes the word lists from the configuration. Five examples are printed side by side so you can see what each model is allowed to read.

In [ ]:
OUT_RE = re.compile(rf"\b({OUTCOME_WORDS})\b", re.I)
MECH_RE = re.compile(rf"\b({MECHANISM_WORDS})\b", re.I)

def clean_spaces(t):
    return re.sub(r"\s+", " ", re.sub(r"\s+([,.;])", r"\1", t)).strip()

def outcome_masked(t):
    # Remove words that reveal the injury or outcome.
    return clean_spaces(OUT_RE.sub(" ", t))

def context_only(t):
    # Remove outcome words AND words that say how the accident happened.
    return clean_spaces(MECH_RE.sub(" ", OUT_RE.sub(" ", t)))

df["text_full"] = df.narrative
df["text_outcome_masked"] = df.narrative.map(outcome_masked)
df["text_context_only"] = df.narrative.map(context_only)
TEXT_VERSIONS = {"full": "text_full", "outcome-masked": "text_outcome_masked", "context-only": "text_context_only"}

examples = df.sample(5, random_state=SEED)[["text_full", "text_outcome_masked", "text_context_only"]]
for _, r in examples.iterrows():
    print("FULL   :", r.text_full[:220]); print("OUTCOME:", r.text_outcome_masked[:220]); print("CONTEXT:", r.text_context_only[:220]); print("-" * 80)

## 7. Split by year and oversample rare accident types

- **Train** ≤ 2021, **validation** 2022, **test** ≥ 2023.
- Rare accident types (e.g. violence, fire) are **randomly duplicated** in the **training set only** up to `OVERSAMPLE_MIN` rows. Paper [6] found random oversampling helped all its models, while SMOTE hurt.

In [ ]:
train_df = df[df.year <= TRAIN_MAX_YEAR].reset_index(drop=True)
val_df = df[df.year == VAL_YEAR].reset_index(drop=True)
test_df = df[df.year > VAL_YEAR].reset_index(drop=True)
split_sizes = {"train": len(train_df), "val": len(val_df), "test": len(test_df)}
print(split_sizes)

def oversample(d, col="event_group", minimum=OVERSAMPLE_MIN, seed=SEED):
    # Duplicate rows of rare classes (random oversampling) until each class has at least `minimum` rows.
    parts = []
    for _, g in d.groupby(col):
        parts.append(g)
        if len(g) < minimum:
            parts.append(g.sample(minimum - len(g), replace=True, random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

train_os = oversample(train_df)
print("Training rows after oversampling:", len(train_os))
# Table: number of reports per accident type in each split (for the report's dataset section)
all_splits = pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")], ignore_index=True)
split_table = pd.crosstab(all_splits.event_group, all_splits.split)[["train", "val", "test"]]
split_table.to_csv(RES / "osha_split_by_event.csv"); print(split_table)

## 8. Experiment E5a · Baseline: TF-IDF + logistic regression

A simple, fast baseline for every output and text version. TF-IDF turns text into word-importance numbers, and logistic regression learns from them. `class_weight="balanced"` gives rare classes more weight. **Macro-F1** is the main score because it treats rare and common classes equally [6].

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report

def log_scores(exp, run, model, version, split, target, y_true, y_pred, notes=""):
    # Log macro-F1, weighted-F1 and accuracy for one output.
    for metric, value in [("macro_F1", f1_score(y_true, y_pred, average="macro")),
                          ("weighted_F1", f1_score(y_true, y_pred, average="weighted")),
                          ("accuracy", accuracy_score(y_true, y_pred))]:
        log_result(exp, run, model, version, "OSHA-SIR construction", split, target, metric, value, notes)

for version, col in TEXT_VERSIONS.items():
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
    Xtr = vec.fit_transform(train_os[col]); Xte = vec.transform(test_df[col])
    for target in TARGETS:
        clf = LogisticRegression(max_iter=2000, class_weight="balanced")
        clf.fit(Xtr, train_os[target])
        log_scores("E5", f"E5a_tfidf_{version}", "TF-IDF+LogReg", version, "test", target, test_df[target], clf.predict(Xte))
    print("done:", version)

e5a = read_log().query("run.str.startswith('E5a') and metric == 'macro_F1'", engine="python").pivot_table(index="text_version", columns="target", values="value")
print(e5a)

## 9. Multi-task transformer (DistilBERT with 3 output heads)

One shared text encoder and three small classification heads (accident type, body part, severity). The losses are added, so the model learns all three tasks at once. The best epoch is chosen on the **validation** year by the average macro-F1 of the three outputs. Written as a plain PyTorch loop so each step is visible and commented.

In [ ]:
import numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

DEVICE_T = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
LABEL2ID = {t: {l: i for i, l in enumerate(ls)} for t, ls in TARGETS.items()}

class ReportDataset(Dataset):
    # Tokenised narratives plus the three integer labels.
    def __init__(self, d, col, tok):
        self.enc = tok(list(d[col]), truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        self.y = {t: torch.tensor([LABEL2ID[t][v] for v in d[t]]) for t in TARGETS}
    def __len__(self):
        return len(self.y["severity"])
    def __getitem__(self, i):
        return {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i],
                **{t: self.y[t][i] for t in TARGETS}}

class MultiTaskClassifier(nn.Module):
    # Shared encoder + one linear head per output (uses the first-token representation).
    def __init__(self, name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleDict({t: nn.Linear(hidden, len(ls)) for t, ls in TARGETS.items()})
    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        h = self.dropout(h)
        return {t: head(h) for t, head in self.heads.items()}

def predict(model, loader):
    # Return predicted label ids and probabilities for every output.
    model.eval(); preds = {t: [] for t in TARGETS}; probs = {t: [] for t in TARGETS}
    with torch.no_grad():
        for b in loader:
            out = model(b["input_ids"].to(DEVICE_T), b["attention_mask"].to(DEVICE_T))
            for t in TARGETS:
                p = out[t].softmax(-1).cpu()
                probs[t].append(p); preds[t].append(p.argmax(-1))
    return {t: torch.cat(preds[t]).numpy() for t in TARGETS}, {t: torch.cat(probs[t]).numpy() for t in TARGETS}

def train_multitask(model_name, version, run_name):
    # Fine-tune one multi-task model on one text version; return (model, tokenizer, test predictions).
    col = TEXT_VERSIONS[version]
    tok = AutoTokenizer.from_pretrained(model_name)
    tr = DataLoader(ReportDataset(train_os, col, tok), batch_size=BATCH, shuffle=True)
    va = DataLoader(ReportDataset(val_df, col, tok), batch_size=BATCH * 2)
    te = DataLoader(ReportDataset(test_df, col, tok), batch_size=BATCH * 2)
    model = MultiTaskClassifier(model_name).to(DEVICE_T)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * len(tr) * EPOCHS), len(tr) * EPOCHS)
    loss_fn = nn.CrossEntropyLoss()
    best_score, best_state = -1, None
    for epoch in range(EPOCHS):
        model.train()
        for step, b in enumerate(tr):
            out = model(b["input_ids"].to(DEVICE_T), b["attention_mask"].to(DEVICE_T))
            loss = sum(loss_fn(out[t], b[t].to(DEVICE_T)) for t in TARGETS)   # equal weight per task
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        vp, _ = predict(model, va)
        score = np.mean([f1_score(val_df[t].map(LABEL2ID[t]), vp[t], average="macro") for t in TARGETS])
        print(f"[{run_name}] epoch {epoch + 1}: validation mean macro-F1 = {score:.4f}")
        if score > best_score:
            best_score, best_state = score, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    tp, tprob = predict(model, te)
    for t in TARGETS:
        inv = {i: l for l, i in LABEL2ID[t].items()}
        log_scores("E5", run_name, model_name, version, "test", t, test_df[t], [inv[i] for i in tp[t]],
                   notes=f"best val mean macro-F1={best_score:.4f}")
    return model, tok, tp, tprob

## 10. Experiments E5b and E5c · DistilBERT on full text and on context-only text

- **E5b (full text)** is the version comparable with paper [6] (macro-F1 0.926 for accident type, ensemble; 0.912 for DistilBERT alone). Their labels were hand-made on a random split, so treat this as context, not a race.
- **E5c (context-only)** is the honest test for our use case, and the model saved for the app.

In [ ]:
_, _, _, _ = train_multitask(MODEL_MAIN, "full", "E5b_distilbert_full")
app_model, app_tok, app_pred, app_prob = train_multitask(MODEL_MAIN, "context-only", "E5c_distilbert_context_only")

## 11. Experiment E5d (optional) · Other transformers

Paper [14] found RoBERTa slightly better than BERT on OSHA reports. Add model names to `EXTRA_MODELS` to compare on the context-only text.

In [ ]:
for m in EXTRA_MODELS:
    short = m.split("/")[-1]
    train_multitask(m, "context-only", f"E5d_{short}_context_only")
if not EXTRA_MODELS:
    print("No extra models configured (EXTRA_MODELS = []).")

## 12. Compare all E5 runs (table, chart and confusion matrix for the report)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

e5 = read_log().query("experiment == 'E5' and metric == 'macro_F1'")
e5_table = e5.pivot_table(index=["run", "model", "text_version"], columns="target", values="value").reset_index()
e5_table["mean_macro_F1"] = e5_table[list(TARGETS)].mean(axis=1)
print(e5_table.to_string())

ax = e5_table.set_index("run")[list(TARGETS)].plot(kind="bar", figsize=(11, 4), rot=30)
ax.axhline(0.926, ls="--", color="grey"); ax.text(0, 0.935, "paper [6]: 0.926 (accident type, full text)", fontsize=8)
ax.set_ylabel("macro-F1 (test, newest years)"); ax.set_ylim(0, 1); ax.set_title("E5: accident-text classifier")
plt.tight_layout(); plt.savefig(FIGS / "fig_E5_macroF1.png", dpi=150); plt.show()

inv = {i: l for l, i in LABEL2ID["event_group"].items()}
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(test_df.event_group, [inv[i] for i in app_pred["event_group"]],
                                        xticks_rotation=60, ax=ax, colorbar=False, normalize="true", values_format=".2f")
ax.set_title("E5c accident type (context-only), normalised by true class")
plt.tight_layout(); plt.savefig(FIGS / "fig_E5c_confusion_event.png", dpi=150); plt.show()
print(classification_report(test_df.event_group, [inv[i] for i in app_pred["event_group"]], zero_division=0))

## 13. Save the app model (context-only DistilBERT + label lists)

Saved in a form the backend can load: encoder and tokenizer (`save_pretrained`), the three heads (`heads.pt`) and the label lists (`labels.json`).

In [ ]:
app_dir = MODELS / "accident_classifier"
app_dir.mkdir(parents=True, exist_ok=True)
app_model.encoder.save_pretrained(app_dir / "encoder"); app_tok.save_pretrained(app_dir / "encoder")
torch.save(app_model.heads.state_dict(), app_dir / "heads.pt")
json.dump({"targets": TARGETS, "max_len": MAX_LEN, "text_version": "context-only", "base_model": MODEL_MAIN,
           "outcome_words": OUTCOME_WORDS, "mechanism_words": MECHANISM_WORDS}, open(app_dir / "labels.json", "w"), indent=2)
print("Saved app model to", app_dir)

## 14. Similar-accident search index (N3)

Every construction report from the **train and validation years** goes into a search index. The app shows the **3 most similar real accidents** for the situation in the photo. Two ways to search:
- **Keyword (BM25):** classic word matching,
- **Meaning (embeddings):** each narrative becomes a vector with `bge-small`; FAISS finds the closest vectors.

The index stores the **original** narrative for display, but search runs on the **context-only** text, because the app's query is also a context description.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

corpus = pd.concat([train_df, val_df]).reset_index(drop=True)
tokenize = lambda t: re.findall(r"[a-z]+", t.lower())

bm25 = BM25Okapi([tokenize(t) for t in corpus.text_context_only])
embedder = SentenceTransformer(EMB_MODEL, device=DEVICE_T)
corpus_emb = embedder.encode(list(corpus.text_context_only), batch_size=128, normalize_embeddings=True, show_progress_bar=True).astype("float32")
index = faiss.IndexFlatIP(corpus_emb.shape[1]); index.add(corpus_emb)          # inner product = cosine (vectors are normalised)

def search_dense(queries, k=3):
    q = embedder.encode(list(queries), normalize_embeddings=True).astype("float32")
    return index.search(q, k)[1]

def search_bm25(query, k=3):
    return np.argsort(-bm25.get_scores(tokenize(query)))[:k]
print("Index size:", index.ntotal)

## 15. Experiment E7 · Are the retrieved accidents relevant?

Automatic check with no manual labelling: each **test-year** report (context-only text) is used as a query. A retrieved accident counts as **relevant** if it has the **same accident type** (lenient) or the **same accident type and body-part group** (strict). Score = **precision@3**. Up to 2,000 test queries.

In [ ]:
queries = test_df.sample(min(2000, len(test_df)), random_state=SEED).reset_index(drop=True)
dense_hits = search_dense(queries.text_context_only, k=3)

def precision_at_3(hit_lists):
    same_type = np.mean([np.mean([corpus.event_group[j] == q.event_group for j in hits]) for hits, (_, q) in zip(hit_lists, queries.iterrows())])
    same_both = np.mean([np.mean([(corpus.event_group[j] == q.event_group) and (corpus.body_group[j] == q.body_group) for j in hits])
                         for hits, (_, q) in zip(hit_lists, queries.iterrows())])
    return same_type, same_both

bm25_hits = [search_bm25(t) for t in queries.text_context_only]
rand_hits = [np.random.RandomState(i).choice(len(corpus), 3, replace=False) for i in range(len(queries))]
e7_rows = []
for name, hits in [("random (floor)", rand_hits), ("BM25 keyword", bm25_hits), ("bge-small embeddings", list(dense_hits))]:
    st, sb = precision_at_3(hits)
    e7_rows.append({"method": name, "P@3 same accident type": st, "P@3 same type + body part": sb})
    log_result("E7", f"E7_{name.split()[0]}", name, "context-only", "OSHA-SIR construction", "test", "event_group", "precision_at_3", st)
    log_result("E7", f"E7_{name.split()[0]}", name, "context-only", "OSHA-SIR construction", "test", "event+body", "precision_at_3", sb)
e7 = pd.DataFrame(e7_rows); print(e7.to_string())

## 16. Try it: a situation described from a photo

This is the kind of text the app's incident writer (N1) produces. The classifier predicts the likely accident type, body part and severity, and the search shows the 3 closest real accidents.

In [ ]:
def analyse_situation(text, k=3):
    # Classifier probabilities for the three outputs + the k most similar past accidents.
    enc = app_tok([context_only(text)], truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE_T)
    app_model.eval()
    with torch.no_grad():
        out = app_model(enc["input_ids"], enc["attention_mask"])
    result = {}
    for t in TARGETS:
        p = out[t].softmax(-1)[0].cpu().numpy()
        top = np.argsort(-p)[:3]
        result[t] = [(TARGETS[t][i], round(float(p[i]), 3)) for i in top]
    hits = search_dense([context_only(text)], k=k)[0]
    result["similar_accidents"] = [corpus.narrative[j][:300] for j in hits]
    return result

demo = analyse_situation("Worker without a hard hat standing on a scaffold below a crane lifting steel beams.")
print(json.dumps(demo, indent=1))

## 17. Save results, index and model (Excel + ZIP)

In [ ]:
import shutil
faiss.write_index(index, str(MODELS / "similar_accidents.faiss"))
corpus[["narrative", "event_group", "body_group", "severity", "year", "event_title", "body_title"]].to_parquet(MODELS / "similar_accidents_meta.parquet")
json.dump({"emb_model": EMB_MODEL, "text_version": "context-only"}, open(MODELS / "similar_accidents_config.json", "w"))

log = read_log()
xlsx = RES / "E5_E7_accident_text.xlsx"
with pd.ExcelWriter(xlsx) as xw:
    e5_table.to_excel(xw, sheet_name="E5", index=False)
    e7.to_excel(xw, sheet_name="E7", index=False)
    label_counts.to_frame().to_excel(xw, sheet_name="label_counts")
    pd.DataFrame([split_sizes]).to_excel(xw, sheet_name="split_sizes", index=False)
    log.to_excel(xw, sheet_name="full_log", index=False)
json.dump({"split": {"train": f"<= {TRAIN_MAX_YEAR}", "val": VAL_YEAR, "test": f"> {VAL_YEAR}"}, "split_sizes": split_sizes,
           "model": MODEL_MAIN, "epochs": EPOCHS, "lr": LR, "batch": BATCH, "max_len": MAX_LEN,
           "oversample_min": OVERSAMPLE_MIN, "emb_model": EMB_MODEL, "seed": SEED}, open(OUT / "run_config_nb3.json", "w"), indent=2)
archive = shutil.make_archive(str(OUT.parent / "safevision_nb3_outputs"), "zip", OUT)
print("Saved", xlsx, "| ZIP:", archive)

In [ ]:
# [colab-only]
from google.colab import files
files.download(archive)
try:
    from google.colab import userdata
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.upload_folder(folder_path=str(OUT), repo_id=f"{api.whoami()['name']}/safevision-artifacts", path_in_repo="nb3_accident_text")
    print("Uploaded to your private HF repo.")
except Exception as e:
    print("HF upload skipped:", e)

## References

Numbering matches *SafeVision_Paper_Summaries.pdf*. Papers used directly here are **in bold**.

5. **Ou, Z., Li, D., Tan, Z., Li, W., Liu, H., & Song, S. (2025). Building safer sites: A large-scale multi-level dataset for construction safety research. arXiv:2508.09203.** (language models beat classic ML on OSHA narratives)
6. **Wang, J., Wu, S., Qu, Z., Shen, J., Memon, S. A., Skitmore, M., & Ma, L. (2025). Classifying OSHA construction accident reports: Leveraging ensemble learning and lightweight large language models (l-LLMs). *Engineering, Construction and Architectural Management*. https://doi.org/10.1108/ECAM-08-2025-1273** (8 OIICS accident types, DistilBERT, random oversampling, macro-F1 0.926; time-based split as future work)
7. **Cao, Y., Qu, Z., Wu, S., Chen, Y., Skitmore, M., Ma, X., & Wang, J. (2026). Analyzing OSHA construction accident reports using BERTopic topic modeling for thematic insights. *Buildings*, 16(1), 10. https://doi.org/10.3390/buildings16010010** (OIICS accident groups on OSHA narratives)
14. **Sadick, A.-M., Smith, B., Nasirzadeh, F., Sadeghi, S., Ayal, S., & Bouadjenek, M. R. (2025). Comparative analysis of BERT and RoBERTa for construction site incident report classification. *CIB Conferences*, 1, 248. https://doi.org/10.7771/3067-4883.1839** (RoBERTa vs BERT)
15. **Cui, Y., et al. (2026). SafeBuild-Bench: A temporal-robust construction safety benchmark with graph-enhanced data mining. arXiv:2608.00068.** (results drift over time, hence the year-based split)
- U.S. Occupational Safety and Health Administration. *Severe Injury Reports* (2015-present). https://www.osha.gov/severe-injury-reports (public domain)
- Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). DistilBERT, a distilled version of BERT. arXiv:1910.01108.
- Xiao, S., Liu, Z., Zhang, P., & Muennighoff, N. (2023). C-Pack: Packaged resources to advance general Chinese embedding (BGE models). arXiv:2309.07597.
- Robertson, S., & Zaragoza, H. (2009). The probabilistic relevance framework: BM25 and beyond. *Foundations and Trends in Information Retrieval*, 3(4), 333-389.
- Johnson, J., Douze, M., & Jégou, H. (2019). Billion-scale similarity search with GPUs (FAISS). *IEEE Transactions on Big Data*, 7(3), 535-547.